# NBA Chat with Data — AI-Powered Basketball Analytics

Launch a full-featured **Chainlit** chat interface to explore 160+ tables of NBA data
using natural language, SQL, Python, and interactive visualizations.

**Features:**
- 7 LLM providers (OpenAI, Anthropic, Google, Ollama, LM Studio, GitHub Copilot, Custom)
- Interactive Plotly & matplotlib charts with team colors
- Advanced metrics (TS%, eFG%, Usage Rate, Pace, Net Rating)
- SQL sandbox + Python execution environment
- Export results as CSV, XLSX, or JSON with one click
- Copy SQL/Python code from any step, or export full session as a script
- Web search for latest NBA news
- Settings panel, chat profiles, action buttons

**Requirements:** An API key from any supported provider.

---

In [ ]:
# Cell 1: Install dependencies
import subprocess, sys

deps = [
    "chainlit>=2.10", "deepagents", "langchain-mcp-adapters",
    "langchain-core>=0.3", "langchain-openai>=0.3",
    "langchain-anthropic>=0.3", "langchain-google-genai>=2.0",
    "mcp[cli]", "pydantic-settings>=2.13",
    "plotly>=6.0", "matplotlib>=3.10", "pandas>=2.2",
    "duckdb>=1.4", "httpx>=0.28", "duckduckgo-search>=7.0",
    "trafilatura>=2.0", "loguru>=0.7", "openpyxl>=3.1", "cloudflared",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", *deps],
    stdout=subprocess.DEVNULL,
)
print("Dependencies installed.")

In [ ]:
# Cell 2: Configure provider + API key
import getpass, json
from pathlib import Path

# ── Provider Selection ──
PROVIDER = "openai"   # Change to: "anthropic", "google", "ollama", "lmstudio", "copilot", "custom"
MODEL = "gpt-4o"      # Change to: "claude-sonnet-4-20250514", "gemini-2.0-flash", etc.

# ── API Key (Kaggle secrets or manual input) ──
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    _key_names = {
        "openai": "OPENAI_API_KEY",
        "anthropic": "ANTHROPIC_API_KEY",
        "google": "GOOGLE_API_KEY",
    }
    API_KEY = _secrets.get_secret(_key_names.get(PROVIDER, "API_KEY"))
    print(f"Loaded {PROVIDER} key from Kaggle secrets")
except Exception:
    API_KEY = getpass.getpass(f"Enter your {PROVIDER} API key: ")

# ── Write config to ~/.nbadb/chat.json ──
config_dir = Path.home() / ".nbadb"
config_dir.mkdir(parents=True, exist_ok=True)
config = {"provider": PROVIDER, "model": MODEL, "api_key": API_KEY}
(config_dir / "chat.json").write_text(json.dumps(config))
print(f"Config saved: provider={PROVIDER}, model={MODEL}")

In [ ]:
# Cell 3: Ensure NBA database is available
import shutil
from pathlib import Path

# Kaggle attaches the dataset at /kaggle/input/basketball/
KAGGLE_DB = Path("/kaggle/input/basketball/nba.duckdb")
LOCAL_DB = Path.home() / ".nbadb/data/nba.duckdb"

if KAGGLE_DB.exists() and not LOCAL_DB.exists():
    LOCAL_DB.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(KAGGLE_DB, LOCAL_DB)
    print(f"Copied database to {LOCAL_DB}")
elif LOCAL_DB.exists():
    print(f"Database ready at {LOCAL_DB}")
else:
    print("Database not found — will download from Kaggle on first chat")

In [ ]:
# Cell 4: Clone nbadb repo (for the chat app code)
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/wyattowalsh/nbadb.git"

# Detect environment
IN_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()
CLONE_DIR = WORK_DIR / "nbadb"
CHAT_DIR = CLONE_DIR / "apps" / "chat"

if not CLONE_DIR.exists():
    print("Cloning nbadb repository...")
    subprocess.check_call(
        ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    print("Repository cloned.")
else:
    print(f"Repository already at {CLONE_DIR}")

os.chdir(str(CHAT_DIR))
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Cell 5: Launch Chainlit + tunnel
import subprocess, sys, threading, time
from pathlib import Path
from IPython.display import display, HTML

PORT = 8421

def _run_chainlit():
    """Run Chainlit server in background."""
    subprocess.run([
        sys.executable, "-m", "chainlit", "run", "chainlit_app.py",
        "--host", "0.0.0.0", "--port", str(PORT),
    ])

# Start server in background thread
thread = threading.Thread(target=_run_chainlit, daemon=True)
thread.start()
time.sleep(5)
print(f"Chainlit server started on port {PORT}")

# Create tunnel for remote environments, direct link for local
IN_KAGGLE = Path("/kaggle").exists()
IN_COLAB = "google.colab" in sys.modules

if IN_KAGGLE or IN_COLAB:
    try:
        from cloudflared import run as cf_run
        tunnel_url = cf_run("localhost", PORT)
        display(HTML(
            f'<h2 style="color:#1D428A">'
            f'Open NBA Chat: <a href="{tunnel_url}" target="_blank">{tunnel_url}</a>'
            f'</h2>'
        ))
    except Exception as e:
        print(f"Tunnel setup failed: {e}")
        print(f"If running locally, open http://localhost:{PORT}")
else:
    url = f"http://localhost:{PORT}"
    display(HTML(
        f'<h2 style="color:#1D428A">'
        f'Open NBA Chat: <a href="{url}" target="_blank">{url}</a>'
        f'</h2>'
    ))

## Usage

1. Click the link above to open the chat interface
2. Use the **gear icon** to switch providers, models, and temperature
3. Pick a **profile** (Quick Stats, Deep Analysis, Visualization)
4. Ask anything about NBA data!

### Example questions

- "Who are the top 10 scorers this season? Show a bar chart with team colors."
- "Compare LeBron and Curry's career TS% trends"
- "What's the correlation between pace and offensive rating?"
- "Plot a scatter of usage rate vs points for all 20+ PPG scorers"
- "Analyze the top 5 MVP candidates across all advanced metrics"

### Troubleshooting

| Issue | Fix |
| --- | --- |
| Tunnel not working | Re-run Cell 5 |
| API key error | Re-run Cell 2 with the correct key |
| Database missing | Ensure the `wyattowalsh/basketball` dataset is attached |
| Import errors | Re-run Cell 1 to reinstall dependencies |

---

*Built with [nbadb](https://github.com/wyattowalsh/nbadb) — the open-source NBA data platform.*